# Fairness methodologies

This notebook shows simple usage of the fairness algorithms implemented.

In [2]:
import numpy as np
from credit_pipeline.data import load_dataset
from credit_pipeline import training
from credit_pipeline.evaluate import get_fairness_metrics
from credit_pipeline.fairness_models import Reweighing, FairGBM, ThresholdOpt
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split

/home/hiaac/Documents/code/credit_pipeline/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Loading data and fitting a baseline model

We will use the German dataset with the gender as the sensitive attribute. The baseline algorithm and the fairness algrithms that employ a base estimator will use LGBM with default hyperparameters.

In [25]:
dataset_name = "german"
seed = 0
df = load_dataset(dataset_name)
X = df.drop(columns=["DEFAULT"])
Y = df["DEFAULT"]
A = (X.Gender == "Male").astype(np.int32)
X = X.drop(columns=["Gender"])

X_train, X_test, Y_train, Y_test, A_train, A_test = train_test_split(X, Y, A, test_size=0.2, random_state=seed)

In [26]:
# fit a baseline model
pipeline = training.create_pipeline(
    X_train, Y_train, classifier=LGBMClassifier(verbose=-1), cat_cols="infer"
)
pipeline.fit(X_train, Y_train);

In [27]:
models_dict = {}
models_dict["baseline"] = pipeline.predict(X_test)

## Pre-processing

In [28]:
rw_model = Reweighing(LGBMClassifier(verbose = -1))
pipeline = training.create_pipeline(X_train, Y_train, classifier = rw_model)
pipeline.fit(X_train, Y_train, classifier__sensitive_attributes=A_train);
models_dict["reweighing"] = pipeline.predict(X_test)

## In-processing

In [29]:
fairgbm_model = FairGBM()
pipeline = training.create_pipeline(X_train, Y_train, classifier = fairgbm_model)
pipeline.fit(X_train, Y_train, classifier__sensitive_attributes=A_train);
models_dict["fairgbm"] = pipeline.predict(X_test)

## Post-processing

In [30]:
post_model = ThresholdOpt(LGBMClassifier(verbose = -1), constraint_value=0.05)
pipeline = training.create_pipeline(X_train, Y_train, classifier = post_model)
pipeline.fit(X_train, Y_train, classifier__sensitive_attributes=A_train);
models_dict["threshold_opt"] = pipeline.predict(X_test, sensitive_attributes=A_test)

## Evaluating models

In [31]:
get_fairness_metrics(
    models_dict,
    X_test,
    Y_test,
    A_test,
)

,model,DPD,EOD,AOD,APVD,GMA,balanced_accuracy
0,baseline,-0.090993,-0.041667,-0.049915,-0.037079,0.763662,0.703717
1,reweighing,-0.045956,0.010965,-0.000895,-0.070821,0.771744,0.716223
2,fairgbm,-0.022059,0.010965,0.019003,-0.101171,0.791296,0.727092
3,threshold_opt,-0.032169,-0.048246,-0.011113,-0.115636,0.776123,0.671809


The baseline model presented 0.09 of demographic parity difference, which is a very high value. This was reduced to 0.04 from the reweighing approach and 0.02 from FairGBM and 0.03 to the threshold optimization. Interestengly, in this example, FairGBM obtained the highest balanced accuracy and threshold optimization the lowest balanced accuracy.